# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a Croissant-standard dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema and is accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and review high-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}")

## 2. Data Overview
Explore what record sets are available in the dataset, including their `@id`, field IDs, and their labels. All references use `@id`s for programmatic reproducibility.

In [ ]:
# List all record sets, fields, and columns by their @id
print("Available record sets in this dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - Record Set @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    # List fields
    if 'field' in rs:
        print("    Fields:")
        for f in rs['field']:
            if isinstance(f, dict):
                print(f"      - Field @id: {f.get('@id', 'N/A')} | name: {f.get('name', 'N/A')}")
            else:
                print(f"      - Field @id: {f}")
    # List columns
    if 'column' in rs:
        print("    Columns:")
        for c in rs['column']:
            if isinstance(c, dict):
                print(f"      - Column @id: {c.get('@id', 'N/A')} | name: {c.get('name', 'N/A')}")
            else:
                print(f"      - Column @id: {c}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame. Use the record set and field `@id`s to ensure accurate access.

In [ ]:
# For demonstration, we'll load all record sets. You can restrict or select one by @id as needed.
dataframes = {}
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f"Loading data for record set: {rs_id}")
    # Load all records for this record set
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        if not df.empty:
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records with columns: {df.columns.tolist()}")
        else:
            print(f"  No records found for {rs_id}.")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

# Print out the first dataframe for exploration purposes
if dataframes:
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in primary record set ({primary_record_set_id}):")
    print(dataframes[primary_record_set_id].columns.tolist())
    dataframes[primary_record_set_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's apply some basic EDA on one of the main record sets. All references to columns/fields use their `@id`s.

- Filtering on a numeric column (e.g., a mental health score field)
- Normalization
- Grouping by a demographic field


In [ ]:
# For this dataset, assume the primary record set contains a field '@id': 'phq9_score' representing the PHQ-9 depression score,
# and '@id': 'gender' representing participant gender.

# Replace with actual @id values from the overview if different:
primary_df = dataframes[primary_record_set_id]

numeric_field_id = 'phq9_score'         # Example: replace by actual field @id
group_field_id = 'gender'               # Example: replace by actual field @id

# Check if intended fields are present
if numeric_field_id in primary_df.columns and group_field_id in primary_df.columns:
    # Filter records with PHQ-9 score > threshold
    threshold = 10
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the PHQ-9 score
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by gender (or other demographic field)
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df)
else:
    print(f"DataFrame columns: {primary_df.columns.tolist()}")
    print(f"One or both of the fields ({numeric_field_id}, {group_field_id}) not present.")

## 5. Visualization
Visualize distributions of a mental health score and its grouping by demographic (e.g., gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in primary_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(primary_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (PHQ-9 score, example)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if numeric_field_id in primary_df.columns and group_field_id in primary_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=primary_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. By referencing data entities via their `@id`, you can ensure reproducibility and clarity in data processing workflows. Further steps may include building predictive models, conducting statistical inference, or mapping the findings to public health interventions.